In [1]:
import argparse
import glob
import os
import sys

import numpy as np
import pandas as pd


In [2]:
# ------------------------- CONFIG -------------------------
NODATA = -9999
BASELINE_YEARS = (2001, 2020)
MIN_VALID_FRAC = 0.70     
MIN_BASE_YEARS = 10       
SD_FLOOR = 0.02           
THRESHOLDS = (-1.0, -1.5)
LOYO = True


In [3]:
# ------------------------- load -------------------------
def load_raw(raw_dir):
    files = sorted(glob.glob(os.path.join(raw_dir, "modis_ndvi_*.csv")))
    if not files:
        raise FileNotFoundError(f"No modis_ndvi_*.csv in {raw_dir}")
    df = pd.concat((pd.read_csv(f, dtype={"pid": str}) for f in files), ignore_index=True)
    df = df.replace(NODATA, np.nan)  # Drive exports keep -9999; local downloads already have NaN
    # counts of 0 mean no valid obs -> make sure NDVI is NaN there
    for c in [c for c in df.columns if c.startswith("ndvi_k")]:
        n_col = "n_k" + c[len("ndvi_k"):]
        if n_col in df.columns:
            df.loc[df[n_col].fillna(0) <= 0, c] = np.nan
    print(f"Loaded {len(files)} files, {df['pid'].nunique()} points, years {df['year'].min()}-{df['year'].max()}")
    return df


def to_long(df):
    long = pd.wide_to_long(
        df, stubnames=["ndvi_k", "n_k"], i=["pid", "lga", "zone", "year"], j="k"
    ).reset_index()
    long = long.rename(columns={"ndvi_k": "ndvi", "n_k": "n_obs"})
    long["k"] = long["k"].astype(int)
    long["doy"] = 1 + 16 * long["k"]
    return long


In [4]:
# ------------------------- point QA -------------------------
def filter_points(long, min_valid_frac):
    vf = long.groupby("pid")["ndvi"].apply(lambda s: s.notna().mean()).rename("valid_frac")
    keep = vf[vf >= min_valid_frac].index
    print(f"Points kept: {len(keep)} / {len(vf)} (valid fraction >= {min_valid_frac})")
    print("Valid fraction distribution:\n", vf.describe().round(3).to_string())
    return long[long["pid"].isin(keep)].copy(), vf



In [5]:
# ------------------------- baseline + z -------------------------
def add_baseline_z(long, base_years=BASELINE_YEARS, loyo=LOYO, min_base_years=MIN_BASE_YEARS):
    y0, y1 = base_years
    long = long.copy()
    long["x2"] = long["ndvi"] ** 2

    pool = long[long["year"].between(y0, y1) & long["ndvi"].notna()]
    agg = (
        pool.groupby(["pid", "k"])
        .agg(b_n=("ndvi", "count"), b_s=("ndvi", "sum"), b_ss=("x2", "sum"))
        .reset_index()
    )
    long = long.merge(agg, on=["pid", "k"], how="left")

    n = long["b_n"].astype(float)
    s = long["b_s"].astype(float)
    ss = long["b_ss"].astype(float)

    if loyo:
        own = long["year"].between(y0, y1) & long["ndvi"].notna()
        n = n - own.astype(float)
        s = s - np.where(own, long["ndvi"], 0.0)
        ss = ss - np.where(own, long["x2"], 0.0)

    ok = n >= min_base_years
    mu = s / n
    var = (ss - n * mu**2) / (n - 1)
    sd = np.sqrt(np.clip(var, 0, None))

    long["ndvi_mu"] = np.where(ok, mu, np.nan)
    long["ndvi_sd"] = np.where(ok, sd, np.nan)
    long["base_n"] = n
    sd_eff = np.maximum(long["ndvi_sd"], SD_FLOOR)
    long["z"] = (long["ndvi"] - long["ndvi_mu"]) / sd_eff

    for t in THRESHOLDS:
        col = f"stress_m{int(abs(t) * 10):02d}"
        flag = (long["z"] < t).astype("Int8")
        flag[long["z"].isna()] = pd.NA
        long[col] = flag

    return long.drop(columns=["x2", "b_n", "b_s", "b_ss"])


In [6]:
# ------------------------- QA summaries -------------------------
def summarise(long, out_dir):
    flag = "stress_m10"
    by_year = long.groupby("year")[flag].agg(lambda s: s.dropna().astype(float).mean()).rename("stress_rate")
    by_zone = long.groupby("zone")[flag].agg(lambda s: s.dropna().astype(float).mean()).rename("stress_rate")
    by_k = long.groupby("k")[flag].agg(lambda s: s.dropna().astype(float).mean()).rename("stress_rate")
    valid_k = long.groupby("k")["ndvi"].apply(lambda s: s.notna().mean()).rename("valid_share")

    by_year.to_csv(os.path.join(out_dir, "qa_stress_by_year.csv"))
    by_zone.to_csv(os.path.join(out_dir, "qa_stress_by_zone.csv"))
    pd.concat([by_k, valid_k], axis=1).to_csv(os.path.join(out_dir, "qa_by_window.csv"))

    print("\nStress rate (z < -1.0) by year:\n", by_year.round(3).to_string())
    print("\nStress rate by zone:\n", by_zone.round(3).to_string())
    print("\nBy window (stress rate, valid share):\n", pd.concat([by_k, valid_k], axis=1).round(3).to_string())



In [7]:
# ------------------------- main -------------------------
def build(raw_dir, out_dir, base_years=BASELINE_YEARS, loyo=LOYO, min_valid_frac=MIN_VALID_FRAC):
    os.makedirs(out_dir, exist_ok=True)
    long = to_long(load_raw(raw_dir))
    long, vf = filter_points(long, min_valid_frac)
    vf.to_csv(os.path.join(out_dir, "point_valid_fraction.csv"))
    long = add_baseline_z(long, base_years, loyo)
    long = long.sort_values(["pid", "year", "k"]).reset_index(drop=True)

    try:
        long.to_parquet(os.path.join(out_dir, "label_long.parquet"), index=False)
        print("\nSaved label_long.parquet")
    except Exception as e:  # pyarrow missing
        long.to_csv(os.path.join(out_dir, "label_long.csv"), index=False)
        print(f"\nparquet failed ({e}); saved label_long.csv")

    summarise(long, out_dir)
    return long


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw-dir", required=True)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--base-start", type=int, default=BASELINE_YEARS[0])
    ap.add_argument("--base-end", type=int, default=BASELINE_YEARS[1])
    ap.add_argument("--no-loyo", action="store_true", help="use the plain (non leave-one-year-out) baseline")
    ap.add_argument("--min-valid-frac", type=float, default=MIN_VALID_FRAC)
    a = ap.parse_args(argv)
    return build(a.raw_dir, a.out_dir, (a.base_start, a.base_end), not a.no_loyo, a.min_valid_frac)


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main()

In [7]:
main(["--raw-dir", r"crop_stress\raw", "--out-dir", r"crop_stress\label"])

Loaded 26 files, 8387 points, years 2001.0-2025.0
Points kept: 3185 / 8387 (valid fraction >= 0.7)
Valid fraction distribution:
 count    8387.000
mean        0.667
std         0.100
min         0.265
25%         0.596
50%         0.673
75%         0.742
max         0.869

Saved label_long.parquet

Stress rate (z < -1.0) by year:
 year
2001.0    0.191
2002.0    0.207
2003.0    0.108
2004.0    0.096
2005.0    0.121
2006.0     0.15
2007.0    0.152
2008.0     0.19
2009.0    0.053
2010.0    0.138
2011.0    0.087
2012.0    0.118
2013.0    0.099
2014.0    0.068
2015.0    0.367
2016.0    0.187
2017.0    0.397
2018.0    0.243
2019.0    0.231
2020.0    0.197
2021.0     0.52
2022.0    0.245
2023.0    0.318
2024.0    0.358
2025.0     0.34

Stress rate by zone:
 zone
0    0.177
1    0.175
2    0.212

By window (stress rate, valid share):
     stress_rate  valid_share
k                           
3         0.222        0.959
4         0.208        0.950
5         0.206        0.925
6         0.205 

,pid,lga,zone,year,k,.geo,crop_frac,system:index,ndvi,n_obs,doy,ndvi_mu,ndvi_sd,base_n,z,stress_m10,stress_m15
0,105,Egbado /Yewa North,0,2001.0,3,NaN,NaN,NaN,0.2377,1.0,49,0.404261,0.101497,18.0,-1.641048,1,1
1,105,Egbado /Yewa North,0,2001.0,4,NaN,NaN,NaN,NaN,NaN,65,0.473429,0.126529,14.0,NaN,<NA>,<NA>
2,105,Egbado /Yewa North,0,2001.0,5,NaN,NaN,NaN,0.4556,1.0,81,0.549996,0.123207,14.0,-0.766164,0,0
3,105,Egbado /Yewa North,0,2001.0,6,NaN,NaN,NaN,0.4462,1.0,97,0.631925,0.118297,14.0,-1.569986,1,1
4,105,Egbado /Yewa North,0,2001.0,7,NaN,NaN,NaN,0.6158,1.0,113,0.704374,0.060957,19.0,-1.453051,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
828095,8385,Atisbo,2,NaN,8,"{""type"":""Point"",""coordinates"":[3.3066062542733...",0.728157,000000000000000020c1,NaN,NaN,129,0.670283,0.055300,18.0,NaN,<NA>,<NA>
828096,8385,Atisbo,2,NaN,9,"{""type"":""Point"",""coordinates"":[3.3066062542733...",0.728157,000000000000000020c1,NaN,NaN,145,0.684650,0.062476,16.0,NaN,<NA>,<NA>
828097,8385,Atisbo,2,NaN,10,"{""type"":""Point"",""coordinates"":[3.3066062542733...",0.728157,000000000000000020c1,NaN,NaN,161,0.715112,0.070134,13.0,NaN,<NA>,<NA>
828098,8385,Atisbo,2,NaN,11,"{""type"":""Point"",""coordinates"":[3.3066062542733...",0.728157,000000000000000020c1,NaN,NaN,177,0.735232,0.110972,11.0,NaN,<NA>,<NA>
